# Preflight and paths

Confirms the exact metadata choice, dataset tree, environment and local DINOv3 availability. No training.

Run the single code cell below with the **Python (skin)** kernel.

In [2]:
from pathlib import Path
import sys

PACKAGE_DIR_INPUT = Path(r"D:\AIUB\Neural\A_Comprehensive_Framework_for_Auditing_Demographic_Bias_in_Synthetic_Skin_Disease_Datasets\revision_package")
if not PACKAGE_DIR_INPUT.is_dir():
    raise FileNotFoundError(f"revision_package not found: {PACKAGE_DIR_INPUT}")
if str(PACKAGE_DIR_INPUT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR_INPUT))

import importlib.util
import json
import os
import platform
import subprocess

required_modules = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "scikit-learn": "sklearn",
    "Pillow": "PIL",
    "jupyterlab": "jupyterlab",
    "ipykernel": "ipykernel",
}
missing = [package for package, module in required_modules.items() if importlib.util.find_spec(module) is None]
if missing:
    command = f'"{sys.executable}" -m pip install -r "{PACKAGE_DIR_INPUT / "requirements-jupyter-minimal.txt"}"'
    raise RuntimeError(f"Missing packages: {missing}\nRun in Anaconda Prompt:\n{command}")

import pandas as pd
from notebook_config import DATA_ROOT, HF_MODEL_SOURCE, METADATA_M, REPO_DIR, RUN_ROOT
from notebook_support import (
    compare_metadata_files,
    select_metadata,
    validate_hierarchical_image_mapping,
    write_json,
)

metadata_path = select_metadata(require_known_hash=True)
comparison = compare_metadata_files()
if comparison.get("metadata_m_exists") and comparison.get("metadata_original_exists"):
    if not comparison.get("equal_after_path_normalization"):
        raise ValueError("The two metadata files differ beyond the expected filepath prefix; stop and inspect them.")
mapping = validate_hierarchical_image_mapping(metadata_path)
frame = pd.read_csv(metadata_path)
required_columns = {"filepath", "split", "label", "age", "dominant_gender", "dominant_race"}
if not required_columns.issubset(frame.columns):
    raise ValueError(f"Missing metadata columns: {sorted(required_columns - set(frame.columns))}")

pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"], capture_output=True, text=True
)
environment = {
    "python_executable": sys.executable,
    "python_version": platform.python_version(),
    "conda_default_env": os.environ.get("CONDA_DEFAULT_ENV", ""),
    "pip_check_ok": pip_check.returncode == 0,
    "pip_check_output": (pip_check.stdout + pip_check.stderr).strip(),
}
try:
    import torch
    environment.update(
        torch_version=torch.__version__,
        cuda_available=bool(torch.cuda.is_available()),
        cuda_device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    )
except Exception as error:
    environment.update(torch_import_error=repr(error), cuda_available=False)

try:
    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained(HF_MODEL_SOURCE, local_files_only=True)
    environment["dinov3_config_available_locally"] = True
    environment["dinov3_hidden_size"] = int(cfg.hidden_size)
except Exception as error:
    environment["dinov3_config_available_locally"] = False
    environment["dinov3_local_check_error"] = str(error)

report = {
    "repository": str(REPO_DIR),
    "metadata_selected": str(metadata_path),
    "preferred_metadata": str(METADATA_M),
    "dataset_root": str(DATA_ROOT),
    "metadata_comparison": comparison,
    "dataset_mapping": mapping,
    "split_counts": frame["split"].value_counts().to_dict(),
    "disease_counts": frame["label"].value_counts().to_dict(),
    "environment": environment,
}
write_json(RUN_ROOT / "preflight" / "preflight_report.json", report)
print(json.dumps(report, indent=2, default=str))
print("\nPASS: use complete_metadata_m.csv. Its relative paths match the Train/Test/Val dataset tree.")
if not environment["pip_check_ok"]:
    print("\nWARNING: pip check reported an environment conflict. Review the output above before model recovery.")


{
  "repository": "D:\\AIUB\\Neural\\A_Comprehensive_Framework_for_Auditing_Demographic_Bias_in_Synthetic_Skin_Disease_Datasets",
  "metadata_selected": "D:\\AIUB\\Neural\\complete_metadata_m.csv",
  "preferred_metadata": "D:\\AIUB\\Neural\\complete_metadata_m.csv",
  "dataset_root": "D:\\AIUB\\Neural\\Synthetic_Skin_Disease_Dataset",
  "metadata_comparison": {
    "metadata_m_exists": true,
    "metadata_original_exists": true,
    "equal_after_path_normalization": true,
    "rows": 565,
    "columns": [
      "filepath",
      "split",
      "label",
      "age",
      "dominant_gender",
      "dominant_race"
    ],
    "metadata_m_sha256": "2ff2c29654ae2966226bff495b951432a7392b0ea92632d6f2726c7f9abb07e6",
    "metadata_original_sha256": "6df1fd913e3905719c74b0c9ea81b1f91bf2d63be01763fdc9ffb17c0ca48025"
  },
  "dataset_mapping": {
    "metadata_rows": 565,
    "metadata_unique_relative_paths": 565,
    "image_files_found": 2750,
    "matched_metadata_images": 565,
    "extra_image_f